# Bias Correction Training (Colab)

Follow these cells top-to-bottom. Use GPU: Runtime → Change runtime type → GPU.


In [ ]:
# 1) Install dependencies
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install xarray netcdf4 tqdm matplotlib


In [ ]:
# 2) Mount Drive and copy project to /content
from google.colab import drive
from pathlib import Path
import shutil, os

drive.mount('/content/drive')

# TODO: set your folder on Drive that contains Bias_Correction_2025 and the NetCDF files
DRIVE_FOLDER = "/content/drive/MyDrive/your_folder"

# Fresh copy of the project
if Path('/content/Bias_Correction_2025').exists():
    shutil.rmtree('/content/Bias_Correction_2025')
shutil.copytree(f"{DRIVE_FOLDER}/Bias_Correction_2025", "/content/Bias_Correction_2025")
%cd /content/Bias_Correction_2025

DATA_PATH  = f"{DRIVE_FOLDER}/GFS_3h_JJAS_india_2019to23_filled.nc"
LABEL_PATH = f"{DRIVE_FOLDER}/IMDAA_3h_JJAS_india_2019to23_filled.nc"
print(DATA_PATH)
print(LABEL_PATH)


In [ ]:
# 3) Update config.py to point to Drive files
from pathlib import Path
cfg = Path('config.py').read_text()
cfg = cfg.replace('DATA_PATH" : "../data/gfs model/GFS_3h_JJAS_india_2019to23_filled.nc', f'DATA_PATH" : "{DATA_PATH}')
cfg = cfg.replace('LABEL_PATH" : "../data/imdaa data truth image/IMDAA_3h_JJAS_india_2019to23_filled.nc', f'LABEL_PATH" : "{LABEL_PATH}')
Path('config.py').write_text(cfg)
print(Path('config.py').read_text())


In [ ]:
# 4) Train
import torch
print('CUDA available:', torch.cuda.is_available())
!python train_bias_correction.py


In [ ]:
# 5) Wire best checkpoint to expected test path
import glob, os, shutil
runs = sorted(glob.glob('training_results_*'))
assert runs, 'No training results found'
best = os.path.join(runs[-1], 'best_model.pth')
os.makedirs('checkpoints/Fixed_Model_Simple', exist_ok=True)
shutil.copy2(best, 'checkpoints/Fixed_Model_Simple/best_model_epoch_50.pth')
print('Copied best checkpoint:', best)


In [ ]:
# 6) Test and display figure
!python test_trained_model_updated.py
from IPython.display import Image, display
import glob
imgs = sorted(glob.glob('figures/BC_test_*.png'))
if imgs:
    display(Image(filename=imgs[-1], width=1000))
else:
    print('No figure found in figures/.')


In [ ]:
# 7) Save artifacts back to Drive (optional)
!mkdir -p "$DRIVE_FOLDER/colab_outputs"
!cp -r figures "$DRIVE_FOLDER/colab_outputs/figures"
!cp -r training_results_* "$DRIVE_FOLDER/colab_outputs/"
!cp checkpoints/Fixed_Model_Simple/best_model_epoch_50.pth "$DRIVE_FOLDER/colab_outputs/"
print('Saved outputs to', f'{DRIVE_FOLDER}/colab_outputs')
